## GAP Pricing Engine — Assumption Master Audit

**Assumption version:** A2026_07  
**Audit generated:** 2026-07-01T14:30:00+00:00  
**Built by:** Mohamed Hussein Gadiwala  
**Source workbook:** Basis and Assumptions.xlsx  
**Source sheet:** Assumptions  

### Executive summary
- **Sheets built:** 10 / 10 in scope
- **Total rows:** 207
- **Snapshot folder:** J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\assumption_snapshots\assumption_A2026_07
- **Master workbook:** assumption_master_A2026_07.xlsx
- **Deferred sheets:** 6 (demand not surfaced in current pricing sheet audit)

Status: **✅ Assumption master construction phase COMPLETE.**

### Guiding principle

The scope of this build was determined by a **formula audit** of 10 pricing sheets: GLA, LSDB, AccDth, CI, Spouse-LSDB, AccDB, Spouses-GLA, Dismemberment, S&C Income, TI. Only assumptions actually referenced by these sheets were built. Everything else was deferred with rationale. This inverts the old "extract everything that might be an assumption" pattern in favour of **demand-driven extraction**.

### Sheet inventory

| # | Sheet | Rows | Structure | Consumed by | Status |
|---|---|---|---|---|---|
| 1 | globals | 10 | key-value | All products (VAT, country codes, Namibia constants) | ✅ |
| 2 | product_limits | 53 | product × country × param | All products (max_age, max_cover, max_multiple, EOH) | ✅ |
| 3 | profit_margins | 9 | product × margin_type | All products (Profit_Margin_* named ranges) | ✅ |
| 4 | conversion_option | 11 | product × age_condition × co_type | GLA, LSDB, CI, Spouse GLA/LSDB, S&C | ✅ |
| 5 | tax_tables | 4 | SA tax gross-up bands | GLA rows 44-49 | ✅ |
| 6 | ctc_options | 3 | option → loading | Spouse GLA row 22, Spouse LSDB row 19 | ✅ |
| 7 | lsdb_features | 87 | feature × cost_basis × key | LSDB rows 22-39, 45-47 | ✅ |
| 8 | spouse_factors | 6 | factor_type × cost_basis × gender | Spouse GLA row 22, Spouse LSDB rows 22-23 | ✅ |
| 9 | ci_table_mapping | 7 | product_option → 4 attributes (wide) | CI rows 33-45 | ✅ |
| 10 | ci_reinstatement | 17 | feature × key | CI rows 27, 57 | ✅ |

### Per-sheet detail

#### 1. globals
- **Rows:** 10
- **Schema:** `param_name, param_value, param_type, source_label, source_occurrence, notes`
- **Values include:** VAT rate (0.15), active country code (1=SA), Namibia competitive factor (-0.35), Namibia stamps duty (0.03), Namibia Covid loading (null — preserved), EAC profit margin (0.06), EAC admin adjustment (1.005), EAC optional cover margin (0.15), EAC max cover (20M), scheme size large threshold (400)
- **Notable gotcha:** Source has TWO VAT cells (0%/15% toggle). Pinned to occurrence=1 (15% cell with "Update this cell..." note).

#### 2. product_limits
- **Rows:** 53
- **Schema:** `product, country, param_name, param_value, param_type, source_label, source_row, notes`
- **Products:** gla, lsdb, ci, ci_existing, ci_new, spouse_gla, spouse_lsdb, ti, adb, accdb, dismem (11 keys)
- **Country dimension:** GLA has SA/NAM/eSw split for 4 params (max_ctc_age, eoh_max, max_multiple, om_reinsurance_retention_limit). Others use `country="all"`.
- **CI split:** Bare `ci` for shared params (retention limit, CO age, WD cap); `ci_existing`/`ci_new` for the variant-specific max_cover / max_multiple / eoh_max_ci.
- **Notable values:** allow_lsdb_ti stored as text 'false' or Python False (varies between openpyxl sessions). LSDB allow_zero_waiting_period and allow_zero_reduction_period stored as string "Y".

#### 3. profit_margins
- **Rows:** 9
- **Schema:** `product, margin_type, margin, param_type, source_label, source_row, notes`
- **Products:** gla, lsdb, ci, ti, adb, accdb, dismem
- **Margin types:** `standard` (7 products), `dc` (only gla and lsdb have DC variants populated)
- **Skipped:** FC, GIP, TIP, SCIC, Ext FC (out of scope per handover)
- **DC blanks:** Products where DC column is blank get NO row emitted (rule 5.5)

#### 4. conversion_option
- **Rows:** 11
- **Schema:** `product, age_condition, co_type, rate, param_type, source_label, source_row, notes`
- **Products:** gla (6 rows including code_map), lsdb (2), ci (1), spouse (2)
- **GLA structure:** 2×2 grid (age × CO type) = 4 rate rows + 2 code_map rows (Retirement+Withdrawal → 1, Withdrawal only → 2)
- **CI oddity:** Source has no `>65` row for CI CO — preserved verbatim (rule 5.5)
- **Spouse:** Uses shared `product="spouse"` key. Both spouse_gla and spouse_lsdb resolve via SP CO in source.

#### 5. tax_tables
- **Rows:** 4
- **Schema:** `country, tax_year, band_index, band_from, basic_amount, margin, primary_rebate, source_row, notes`
- **Country:** SA only. Namibia GIP tax table at rows 458-468 NOT imported (GIP out of scope).
- **Actuarial cross-check:** `basic_amount[N] ≈ basic_amount[N-1] + (band_from[N] - band_from[N-1]) × margin[N-1]` verified within R1 tolerance.
- **Primary rebate:** Not present in this block (only in GIP block).

#### 6. ctc_options
- **Rows:** 3
- **Schema:** `option_key, loading, param_type, source_label, source_row, notes`
- **Options:** synergy_1_yr_payout (0), synergy_2_yr_payout (0), other_disability (0)
- **All zero:** Currently parameter slots. Sheet exists to keep the pricing lookup path stable.
- **Original name:** "multiple_loading" — renamed after diagnostic showed the actual lookup at `$F$222:$G$224` was CTC options, not disability-months multiple loading.

#### 7. lsdb_features
- **Rows:** 87
- **Schema:** `feature, cost_basis, key, value, param_type, source_row, notes`
- **Features (8):**
  - `db_definitions` (8): 4 disability definitions × Basic/AIDS
  - `waiting_periods` (10): 5 periods × Basic/AIDS
  - `instalments` (12): 6 periods × Basic/AIDS
  - `reduction_period` (12): 6 periods × code/factor
  - `backdating` (6): 3 positional slots × Basic/AIDS (all zero placeholders)
  - `lsdb_to_gla_aids` (1): scalar factor 0.6
  - `red_period_to_table` (18): "Red_NRA" code → table number
  - `nra_to_use` (20): actual NRA 50-69 → NRA to use 60/63/65
- **Backdating ambiguity:** Source rows 305-307 have no key column. Positional idx_1/2/3 assigned with note.

#### 8. spouse_factors
- **Rows:** 6
- **Schema:** `factor_type, cost_basis, gender, value, param_type, source_row, notes`
- **Factor types:** basic_spouses_factor (4 rows, M/F × basic/aids), db_definition_factor (2 rows, basic/aids)
- **AIDS variant blank:** Source rows 361-362 for AIDS spouses factor are BLANK. Preserved as null (rule 5.5). Placeholder for future AIDS variant.
- **DB definition factors:** Vertical layout in source (label at row N col B, value at row N+1 col B). Different pattern from most other tables.

#### 9. ci_table_mapping
- **Rows:** 7
- **Schema:** `product_option, accelerator_table, standalone_table, age_definition, variant_type, source_row, notes`
- **Format:** WIDE (not long) — matches VLOOKUP contract of pricing formula
- **Product options:** Cancer_Protect, Core, Essential, Comprehensive, Comprehensive_Plus, Extensive, Extensive_Plus
- **Business rule invariants verified:**
  - ALB ↔ New / ANB ↔ Existing (all 7 rows conform)
  - New products have single unified table (accelerator_table == standalone_table)
  - Existing products have distinct accelerator vs standalone tables

#### 10. ci_reinstatement
- **Rows:** 17
- **Schema:** `feature, key, value, param_type, source_row, notes`
- **Features (4):**
  - `reinstatement_period` (6): text label → days (30d, 60d, 90d, 180d, 12m→365, 24m→730)
  - `reinstatement_proportion` (6): % of rate difference (0.9589 → 0.0)
  - `discount_reduction_factor` (4): 30d→0, 90d→0.125, 180d→0.25, 365d→0.5
  - `product_exclusion` (1): Cancer_Protect excluded from reinstatement logic
- **Vocabulary drift:** discount table uses "365_days" where period/proportion tables use "12_months" for equivalent duration. Preserved verbatim; pricing engine will need a normaliser.
- **Actuarial invariant enforced:** reinstatement_proportion monotonically decreasing.

### Design decisions locked in

- **5.1** Shape-based sheet grouping — one clean shape per sheet where possible
- **5.2** Long format for factor tables (with discriminator column); wide format for lookup registries (ci_table_mapping)
- **5.3** Snake_case standardised column names
- **5.4** Values frozen from source — no rounding, no rewriting, no invented data
- **5.5** Blanks preserved as null — Namibia Covid loading, AIDS spouse factors, DC margins where source was blank
- **5.6** Dual output: readable xlsx + immutable parquet
- **5.7** Append-only audit trail (build_log.csv with SHA-256 per file)
- **5.8** Anchor-based extraction — survives sheet edits (row shifts won't silently break builds)
- **5.9** Diagnostic-first pattern — every sheet started with a Cell 2 diagnostic that mapped label locations before writing the harvester
- **5.10** Frozen expectations dict per sheet — build fails loudly if source drift changes a value
- **5.11** Type classification done in Python before polars upcasts mixed columns to Utf8
- **5.12** Anchor label matching is column-agnostic (search all cols, filter by row range later) — original col-B-only pattern caused silent misses on Namibia block and Scheme Size

### Product key conventions

- **11 in-scope products:** gla, lsdb, ci (+ ci_existing/ci_new variants), spouse_gla, spouse_lsdb, ti, adb, accdb, dismem, sci (not yet consumed), education (params in globals)
- **CI hierarchy:** bare `ci` key holds shared params; `ci_existing` and `ci_new` hold variant-specific limits; granular product options (Cancer_Protect, Core, etc.) resolved via ci_table_mapping
- **Spouse hierarchy:** `spouse_gla` and `spouse_lsdb` are distinct products with own limits; bare `spouse` key used for shared CO rates only
- **Skipped:** FC, GIP, TIP, SCIC, Ext FC (per handover — separate pricing engines exist)

### Gotchas encountered during build

- **VAT toggle** — Source has two VAT cells (0% and 15%). Pinned extraction to occurrence=1 (15%). If VAT rate changes, actuary edits that specific cell.
- **Column layout drift** — Namibia block anchors sit in col E (not B). Scheme Size label at col F. Original code assumed col B for everything. Fixed by making anchor search column-agnostic.
- **Polars type upcast** — Building a DataFrame from mixed-type rows caused polars to upcast the whole `param_value` column to Utf8, which then made `classify()` return "string" for every value. Fix: classify types in Python land BEFORE polars sees the column.
- **Polars null-skipping** — `pl.col.map_elements()` defaults to `skip_nulls=True`. Nulls need explicit `skip_nulls=False` for classify to return "null" type.
- **Text-typed booleans** — TI `allow_lsdb_ti` sometimes returns as Python `False`, sometimes as text string `'false'` — depends on openpyxl session state. Accept both. Rule 5.4.
- **String Y/N flags** — LSDB `allow_zero_waiting_period` and `allow_zero_reduction_period` stored as string "Y". Preserved. Pricing engine needs a Y/N/true/false/0/1 normaliser.
- **VLOOKUP vocabulary drift** — CI reinstatement discount table uses "365 days" for what other CI tables call "12 months". Both preserved verbatim.
- **Backdating factors no key column** — LSDB rows 305-307 have no explicit key. Assigned positional idx_1/2/3 with note.
- **AIDS spouse factor block empty** — Rows 361-362 blank in source. Preserved null. Placeholder for future AIDS variant.
- **Vertical vs horizontal label→value** — Most sheets use horizontal (label col B, value col C). Spouses DB Definition Factors use vertical (label row N, value row N+1). Handled per-anchor.
- **Source typo preserved** — "Admin Ajustment" (should be "Adjustment") at B196. Not corrected. Rule 5.4.
- **EAC section boundary** — Small EAC block at rows 194-201 mostly overlaps with globals content. Rule: params captured in globals not re-emitted in product_limits.

### Named ranges NOT served by this bundle

These pricing named ranges resolve to **basis** (not assumption) sources — captured in a separate build:
- `CI_Standalone_Loading`, `CI_Industry_Loading_New` → `CI Basis` sheet
- `CI_Aids_Factors`, `CI_GLA_Aids`, `CI_Region_Adjustment` → `CI Basis`
- `CI_Salary_Factors_Basic`, `CI_Salary_Factors_Aids` → `CI Basis`
- `AccDB_Basis_Table`, `AccDB_SalaryFactors_Table` → `AccDB Basis`
- `Dismem_Basis_Table`, `Dismem_SalaryFactor_Table` → `Dismemberment Basis`
- `TI_Table_FY`, `TI_Table_Ongoing`, `TI_Table_Spread` → `TI Tables`
- `GLA_Trend_M`, `GLA_Trend_F` → `GLA Basis`
- Various industry / region / age lookup tables → respective `_Basis` sheets

### Deferred sheets (demand not surfaced)

These were in the original 17-sheet plan but not consumed by any of the 10 audited pricing sheets. Rebuild when downstream needs surface:

| Deferred sheet | Rationale | Trigger to build |
|---|---|---|
| `premium_waiver` | GLA rows Z4/AA4 (Waiver of GLA Premium) empty in current pricing | If Waiver formulas materialise |
| `anti_selection` | ORB loading grid (15%/20%/20%) not referenced in member rows | Likely used at scheme aggregation |
| `scheme_fees` | 12-band % of premium not referenced in member rows | Output/aggregation stage |
| `expense_loadings` | Old TSA table not referenced in member rows | Output/aggregation stage |
| `eoh_layers` | Consumed pre-computed from Pricing Outputs sheet | If EOH becomes engine-side |
| `intermediary` | Broker/Rep/Direct codes not referenced in pricing | Commercial routing layer |
| `multiple_loading` (disability months) | Not referenced by any audited pricing sheet | Scheme aggregation? |
| GIP/TIP/FC blocks | Out of scope — separate pricing engines | Never |

### Next steps

Post-construction tasks not part of this build:
- Add manifest.json in assumption_snapshots/assumption_A2026_07/ — effective_date, approved_by, approved_at, sheet inventory, file hashes
- Audit `'GLA - Pricing Outputs'` sheet to confirm deferred sheets aren't consumed there either
- Move to pricing_core/ phase — implement `basis/loader.py` with `load_assumption(version) → AssumptionBundle`
- Build named-range → parquet coordinates map (pricing engine layer)
- Small Y/N/bool normaliser helper in basis loader for text-typed booleans and string flags
- Small CI vocabulary normaliser ("365_days" ↔ "12_months")

_Audit generated 2026-07-01T14:30:00+00:00 by Copilot for Mohamed Hussein Gadiwala._